In [ ]:
import asyncio
import json
import os
import platform
import struct

import ipywidgets
from IPython.display import display

import xyg
from geo_mounted_hierarchy_temporal_fixture import mounted_hierarchy_chart_fixture
from xyg import _native
from xyg._geo_hierarchy import is_hierarchy_frame
from xyg._geo_temporal import _BINDINGS

source, scope, lane, caller_frame, chart = mounted_hierarchy_chart_fixture()
widget = chart.widget(frame=caller_frame, selected_scope=scope, hierarchy_lane=lane)
caller_frame.close()
controller = xyg.TemporalController(instance_id=64, domain=(-(1 << 63), (1 << 63) - 1))
binding = widget.bind_temporal(controller)
controller.set_cursor(3)
output = ipywidgets.Output()
status = ipywidgets.HTML()
display(output, status)
with output:
    display(widget)
kernel_environment = {
    "python": platform.python_version(),
    "nativeAbi": _native._lib.xyg_abi_version(),
    "nativePath": os.environ["XYG_NATIVE_LIB"],
    "packageFile": xyg.__file__,
}

In [ ]:
async def abort_and_release():
    global \
        binding, \
        adapter, \
        caller, \
        source_current, \
        aborted_sequence, \
        accepted_sequence, \
        phases, \
        accepted_camera
    await binding.when_idle()
    assert binding.error is None, binding.error
    adapter = widget._adapter
    for _ in range(200):
        if (
            adapter._frame.data.identity["camera"]["center_x"] != 0
            and adapter._live_candidate.frame is None
        ):
            break
        await asyncio.sleep(0.05)
    accepted_camera = dict(adapter._frame.data.identity["camera"])
    assert accepted_camera["center_x"] != 0, "trusted canvas keyboard pan did not retire"
    assert adapter._frame.data.identity["time"]["instant"] == 3
    assert is_hierarchy_frame(adapter._frame)
    assert adapter._frame.data.visible_vertices == 2
    assert adapter._frame.data.selection["ids"].tolist() == [(1 << 64) - 1]
    source_current = source.current
    accepted_sequence = adapter._sequence
    caller = adapter._frame.retain()
    native_ready = asyncio.Event()
    held = []
    send = widget.send

    def hold(message, buffers=()):
        if (
            not held
            and message.get("type") == "geo_host"
            and buffers
            and bytes(buffers[0][:4]) == b"XYGH"
            and struct.unpack_from("<I", buffers[0], 8)[0] == 6
        ):
            held.append((message, buffers))
            native_ready.set()
        else:
            send(message, buffers=buffers)

    widget.send = hold
    controller.set_cursor(4)
    await asyncio.wait_for(native_ready.wait(), 15)
    widget.send = send
    candidate = adapter._live_candidate.frame
    aborted_sequence = adapter._live_candidate.sequence_floor
    assert aborted_sequence > accepted_sequence
    assert candidate.data.identity["sequence"] == aborted_sequence
    assert is_hierarchy_frame(candidate)
    closing = asyncio.create_task(binding.aclose())
    while not binding._active["invalidated"]:
        await asyncio.sleep(0)
    send(held[0][0], buffers=held[0][1])
    held.clear()
    await closing
    assert adapter._sequence == accepted_sequence
    assert adapter._live_candidate.sequence_floor == aborted_sequence
    assert adapter._frame.data.identity["time"]["instant"] == 3
    try:
        _ = candidate.data
        raise AssertionError("aborted frame remains owned")
    except RuntimeError:
        pass
    phases = [{"time": 3, "sequence": str(accepted_sequence), "aborted": str(aborted_sequence)}]
    output.clear_output(wait=False)
    for _ in range(200):
        if not adapter.mounted:
            break
        await asyncio.sleep(0.05)
    assert not adapter.mounted, "actual Output clear did not release renderer"
    assert not widget._close_requested
    status.value = "XYG aborted hierarchy renderer released"


phase_task = asyncio.create_task(abort_and_release())
phase_task.add_done_callback(
    lambda t: setattr(status, "value", repr(t.exception())) if t.exception() else None
)

In [ ]:
controller.set_cursor(23)
binding = widget.bind_temporal(controller)
with output:
    display(widget)

In [ ]:
async def after_remount():
    global binding
    await binding.when_idle()
    assert binding.error is None, binding.error
    assert adapter._frame.data.identity["camera"] == accepted_camera
    assert adapter._sequence > aborted_sequence
    assert adapter._frame.data.identity["time"]["instant"] == 23
    assert is_hierarchy_frame(adapter._frame)
    assert adapter._frame.data.visible_vertices == 2
    assert adapter._frame.data.selection["ids"].tolist() == [(1 << 64) - 1]
    assert caller.data.identity["time"]["instant"] == 3
    assert source.current is source_current
    phases.append({"time": 23, "sequence": str(adapter._sequence), "visible": 2})
    status.value = "XYG hierarchy remount accepted above aborted sequence"


phase_task = asyncio.create_task(after_remount())
phase_task.add_done_callback(
    lambda t: setattr(status, "value", repr(t.exception())) if t.exception() else None
)

In [ ]:
async def half_open():
    global binding
    await binding.aclose()
    controller.set_range(100000, 100001)
    binding = widget.bind_temporal(controller, "range")
    await binding.when_idle()
    assert binding.error is None, binding.error
    assert adapter._frame.data.identity["camera"] == accepted_camera
    assert is_hierarchy_frame(adapter._frame)
    assert adapter._frame.data.visible_vertices == 0
    assert adapter._frame.data.selection["ids"].tolist() == [(1 << 64) - 1]
    phases.append({"range": [100000, 100001], "visible": 0})
    status.value = "XYG remounted hierarchy empty range accepted"


phase_task = asyncio.create_task(half_open())
phase_task.add_done_callback(
    lambda t: setattr(status, "value", repr(t.exception())) if t.exception() else None
)

In [ ]:
async def restore():
    global binding
    await binding.aclose()
    controller.set_cursor(6)
    binding = widget.bind_temporal(controller)
    await binding.when_idle()
    assert binding.error is None, binding.error
    assert adapter._frame.data.identity["camera"] == accepted_camera
    assert is_hierarchy_frame(adapter._frame)
    assert adapter._frame.data.visible_vertices == 2
    assert adapter._frame.data.selection["ids"].tolist() == [(1 << 64) - 1]
    phases.append({"time": 6, "visible": 2})
    status.value = "XYG remounted hierarchy restored cursor accepted"


phase_task = asyncio.create_task(restore())
phase_task.add_done_callback(
    lambda t: setattr(status, "value", repr(t.exception())) if t.exception() else None
)

In [ ]:
async def finish():
    global hierarchy_report
    await binding.aclose()
    assert source not in _BINDINGS
    selected_ids = [str(int(v)) for v in adapter._frame.data.selection["ids"]]
    caller.close()
    widget.close()
    while adapter.mounted:
        await asyncio.sleep(0.01)
    await widget._geo_rpc_task
    assert adapter._anchor is None
    lane.close()
    source.close()
    scope.close()
    controller.close()
    hierarchy_report = {
        "phases": phases,
        "sourceClaimReleased": True,
        "releaseACK": True,
        "selectedIds": selected_ids,
        "kernelEnvironment": kernel_environment,
        "acceptedCamera": accepted_camera,
    }
    status.value = "XYG remounted hierarchy native cleanup acknowledged"


phase_task = asyncio.create_task(finish())
phase_task.add_done_callback(
    lambda t: setattr(status, "value", repr(t.exception())) if t.exception() else None
)

In [ ]:
print("XYG hierarchy remount report " + json.dumps(hierarchy_report))